# Proyecto: Convocatoria Selección Española — Mundial 2026
## Análisis de datos con Python, MySQL y Power BI
### Autor: Martín López | Unicorn Academy | 2026

In [1]:
import os
import pandas as pd

# =====================================================
# CONFIGURACIÓN DE RUTAS
# Los archivos CSV deben estar en una carpeta 'datos'
# ubicada en el mismo directorio que este notebook.
# =====================================================
DATA_PATH = 'datos'


# 1. Selección de Jugadores españoles con mejor rendimiento individual en las 5 mejores ligas europeas.

### 1.1 Intento de extracción automática desde FBref

#### Intentamos importar datos directamente desde python pero la pagina web nos bloquea el acceso. Por lo que decidimos hacerlo manualmente.
pd.read_html('https://fbref.com/en/comps/Big5/stats/players/Big-5-European-Leagues-Stats#header')

### 1.2 Carga manual del dataset desde CSV

In [2]:
# Copiamos la tabla desde la pagina web y la pegamos en excel, hacemos una pequeña limpieza de filas repetidas y guardamos la tabla
# en formato .csv y procedemos con la importación con pandas.
df = pd.read_csv("datos/players_data-2025_2026.csv", sep=';')
print(df.shape)
print(df.head())

(2724, 26)
   Rk               Player   Nation    Pos              Squad  \
0   1     Brenden Aaronson   us USA  MF,FW       Leeds United   
1   2          Zach Abbott  eng ENG     DF  Nottingham Forest   
2   3  Jones El-Abdellaoui   ma MAR  MF,FW         Celta Vigo   
3   4        Himad Abdelli   dz ALG  FW,MF          Marseille   
4   5        Himad Abdelli   dz ALG     MF             Angers   

                 Comp     Age    Born  MP  Starts  ... PK  PKatt  CrdY  CrdR  \
0  eng Premier League  25-156  2000.0  30      24  ...  0      0     1     0   
1  eng Premier League  19-318  2006.0   2       1  ...  0      0     0     0   
2          es La Liga  20-074  2006.0  19       4  ...  0      0     0     0   
3          fr Ligue 1  26-130  1999.0   4       0  ...  0      0     1     0   
4          fr Ligue 1  26-130  1999.0  13      11  ...  2      2     1     0   

   Gls.1  Ast.1  G+A.1  G-PK.1  G+A-PK  Matches  
0   0.18   0.14   0.32    0.18    0.32  Matches  
1   0.00   0.00  

### 1.3 Exploración de los datos

In [3]:
# Procedemos con la exploración de los datos antes de empezar con la limpieza, cuantas filas y columnas tenemos.
print(df.shape)

(2724, 26)


In [4]:
# Nombres de las columnas, Aqui notamos que están en ingles y algunas solo nos muestran iniciales, que habrán que modificar.
print(df.columns.tolist())

['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'Gls', 'Ast', 'G+A', 'G-PK', 'PK', 'PKatt', 'CrdY', 'CrdR', 'Gls.1', 'Ast.1', 'G+A.1', 'G-PK.1', 'G+A-PK', 'Matches']


In [5]:
# Que tipo de dato tiene cada columna
print(df.dtypes)

Rk           int64
Player      object
Nation      object
Pos         object
Squad       object
Comp        object
Age         object
Born       float64
MP           int64
Starts       int64
Min         object
90s        float64
Gls          int64
Ast          int64
G+A          int64
G-PK         int64
PK           int64
PKatt        int64
CrdY         int64
CrdR         int64
Gls.1      float64
Ast.1      float64
G+A.1      float64
G-PK.1     float64
G+A-PK     float64
Matches     object
dtype: object


In [6]:
# Hay valores nulos?
print(df.isnull().sum())

Rk         0
Player     0
Nation     2
Pos        0
Squad      0
Comp       0
Age        1
Born       1
MP         0
Starts     0
Min        0
90s        0
Gls        0
Ast        0
G+A        0
G-PK       0
PK         0
PKatt      0
CrdY       0
CrdR       0
Gls.1      0
Ast.1      0
G+A.1      0
G-PK.1     0
G+A-PK     0
Matches    1
dtype: int64


In [7]:
# Estadisticas básicas de las columnas numéricas
print(df.describe())

                Rk         Born           MP       Starts          90s  \
count  2724.000000  2723.000000  2724.000000  2724.000000  2724.000000   
mean   1362.500000  1999.651855    15.747063    11.177680    11.145852   
std     786.495391     4.556203     9.091624     9.070378     8.621549   
min       1.000000  1983.000000     1.000000     0.000000     0.000000   
25%     681.750000  1997.000000     7.000000     3.000000     3.200000   
50%    1362.500000  2000.000000    17.000000    10.000000     9.800000   
75%    2043.250000  2003.000000    24.000000    19.000000    18.200000   
max    2724.000000  2009.000000    31.000000    31.000000    31.000000   

               Gls          Ast          G+A         G-PK           PK  \
count  2724.000000  2724.000000  2724.000000  2724.000000  2724.000000   
mean      1.359031     0.938693     2.297724     1.228708     0.130323   
std       2.425146     1.518978     3.394252     2.117943     0.608042   
min       0.000000     0.000000     0

### 1.4 Limpieza de datos

In [8]:
# Podemos observar que el nombre de las columnas están en ingles y solo iniciales, las cambiaremos a un idioma mas entendible al usuario final.
df = df.rename(columns={
    'Rk': 'ranking',
    'Player': 'jugador',
    'Nation':'nacionalidad',
    'Pos':'posicion',
    'Squad':'equipo',
    'Comp':'competicion',
    'Age':'edad',
    'Born':'nacimiento',
    'MP':'partidos_jugados',
    'Starts':'partidos_titular',
    'Min':'minutos_disputados',
    '90s':'partidos_completos',
    'Gls':'goles',
    'Ast':'asistencias',
    'G+A':'goles_+_asistencias',
    'G-PK':'goles_normales',
    'PK':'goles_de_penal',
    'PKatt':'penales_cobrados',
    'CrdY':'tarjetas_amarillas',
    'CrdR':'tarjetas_rojas',
    'Gls.1':'goles_por_partido',
    'Ast.1':'asistencias_por_partido',
    'G+A.1':'goles_y_asistencias_por_partido',
    'G-PK.1':'goles_normales_por_partido',
    'G+A-PK':'goles_normales_y_asistencias',
    'Matches':'partidos'})
print(df.columns.tolist())

['ranking', 'jugador', 'nacionalidad', 'posicion', 'equipo', 'competicion', 'edad', 'nacimiento', 'partidos_jugados', 'partidos_titular', 'minutos_disputados', 'partidos_completos', 'goles', 'asistencias', 'goles_+_asistencias', 'goles_normales', 'goles_de_penal', 'penales_cobrados', 'tarjetas_amarillas', 'tarjetas_rojas', 'goles_por_partido', 'asistencias_por_partido', 'goles_y_asistencias_por_partido', 'goles_normales_por_partido', 'goles_normales_y_asistencias', 'partidos']


In [9]:
# Por buenas prácticas nombramos las columnas con minusculas y en lugar de un espacio el guion bajo.
df = df.rename(columns={
    'Nacionalidad':'nacionalidad',
    'Posicion':'posicion',
    'Equipo':'equipo',
    'Competicion':'competicion',
    'Partidos jugados':'partidos_jugados',
    'Partidos titular':'partidos_titular',
    'Minutos disputados':'minutos_disputados',
    'Partidos completos':'partidos_completos',
    'Goles':'goles',
    'Asistencias':'asistencias',
    'Goles y Asistencias':'goles_asistencias',
    'Goles normales':'goles_normales',
    'Goles de penal':'goles_penalty',
    'Penales cobrados':'penaltys_cobrados',
    'Tarjetas Amarillas':'tarjetas_amarillas',
    'Tarjetas Rojas':'tarjetas_rojas',
    'Goles por partido':'goles_partido',
    'Asistencias por partido':'asistencias_partido',
    'Goles y Asistencias por partido':'goles_asistencias_partido',
    'Goles normales por partido':'goles_normales_partido',
    'Goles normales y Asistencias':'goles_normales_asistencias',
    'Partidos':'partidos'})
print(df.columns.tolist())

['ranking', 'jugador', 'nacionalidad', 'posicion', 'equipo', 'competicion', 'edad', 'nacimiento', 'partidos_jugados', 'partidos_titular', 'minutos_disputados', 'partidos_completos', 'goles', 'asistencias', 'goles_+_asistencias', 'goles_normales', 'goles_de_penal', 'penales_cobrados', 'tarjetas_amarillas', 'tarjetas_rojas', 'goles_por_partido', 'asistencias_por_partido', 'goles_y_asistencias_por_partido', 'goles_normales_por_partido', 'goles_normales_y_asistencias', 'partidos']


In [10]:
# Consultamos nuevamente los datos y podemos observar que minutos_disputados aparece como object y deberia ser numérico
print(df.dtypes)

ranking                              int64
jugador                             object
nacionalidad                        object
posicion                            object
equipo                              object
competicion                         object
edad                                object
nacimiento                         float64
partidos_jugados                     int64
partidos_titular                     int64
minutos_disputados                  object
partidos_completos                 float64
goles                                int64
asistencias                          int64
goles_+_asistencias                  int64
goles_normales                       int64
goles_de_penal                       int64
penales_cobrados                     int64
tarjetas_amarillas                   int64
tarjetas_rojas                       int64
goles_por_partido                  float64
asistencias_por_partido            float64
goles_y_asistencias_por_partido    float64
goles_norma

In [11]:
# Consultamos los nulos que hay en cada columna
print(df.isnull().sum())

ranking                            0
jugador                            0
nacionalidad                       2
posicion                           0
equipo                             0
competicion                        0
edad                               1
nacimiento                         1
partidos_jugados                   0
partidos_titular                   0
minutos_disputados                 0
partidos_completos                 0
goles                              0
asistencias                        0
goles_+_asistencias                0
goles_normales                     0
goles_de_penal                     0
penales_cobrados                   0
tarjetas_amarillas                 0
tarjetas_rojas                     0
goles_por_partido                  0
asistencias_por_partido            0
goles_y_asistencias_por_partido    0
goles_normales_por_partido         0
goles_normales_y_asistencias       0
partidos                           1
dtype: int64


In [12]:
# Eliminamos la columna partidos ya que no nos ofrece ninguna información importante.
df = df.drop(columns=['partidos'])

In [13]:
# Consultamos los nulos con detalle
print(df[df.isnull().any(axis=1)])

      ranking       jugador nacionalidad posicion    equipo competicion  \
1622     1623  Nathan Mbala          NaN       FW      Metz  fr Ligue 1   
2520     2521    Yael Trepy          NaN       FW  Cagliari  it Serie A   

        edad  nacimiento  partidos_jugados  partidos_titular  ...  \
1622  18-036      2008.0                 7                 1  ...   
2520     NaN         NaN                 7                 0  ...   

     goles_normales  goles_de_penal  penales_cobrados  tarjetas_amarillas  \
1622              2               0                 0                   0   
2520              1               0                 0                   0   

      tarjetas_rojas  goles_por_partido  asistencias_por_partido  \
1622               0               1.25                      0.0   
2520               0               1.02                      0.0   

      goles_y_asistencias_por_partido  goles_normales_por_partido  \
1622                             1.25                       

In [14]:
# Dejaremos los nulos como están ya que son nacionalidad edad y nacimiento y son de jugadores no españoles, 
# mientras tanto consultamos como van quedando nuestras columnas
print(df.dtypes)
print(df.shape)

ranking                              int64
jugador                             object
nacionalidad                        object
posicion                            object
equipo                              object
competicion                         object
edad                                object
nacimiento                         float64
partidos_jugados                     int64
partidos_titular                     int64
minutos_disputados                  object
partidos_completos                 float64
goles                                int64
asistencias                          int64
goles_+_asistencias                  int64
goles_normales                       int64
goles_de_penal                       int64
penales_cobrados                     int64
tarjetas_amarillas                   int64
tarjetas_rojas                       int64
goles_por_partido                  float64
asistencias_por_partido            float64
goles_y_asistencias_por_partido    float64
goles_norma

### 1.5 Corrección de tipos de datos

In [15]:
# Podemos observar que la columna minutos_disputados está en formato object y deberemos pasarla a formato numérico
df['minutos_disputados'] = df['minutos_disputados'].str.replace(',', '').astype(int)
print(df.dtypes)

ranking                              int64
jugador                             object
nacionalidad                        object
posicion                            object
equipo                              object
competicion                         object
edad                                object
nacimiento                         float64
partidos_jugados                     int64
partidos_titular                     int64
minutos_disputados                   int64
partidos_completos                 float64
goles                                int64
asistencias                          int64
goles_+_asistencias                  int64
goles_normales                       int64
goles_de_penal                       int64
penales_cobrados                     int64
tarjetas_amarillas                   int64
tarjetas_rojas                       int64
goles_por_partido                  float64
asistencias_por_partido            float64
goles_y_asistencias_por_partido    float64
goles_norma

In [16]:
# La columna edad está en formato 25 - 128 es decir 25 años y 128 dias, pero solo nos interesa la edad.
df['edad'] = df['edad'].str.split('-').str[0].astype(float)
print(df['edad'].head(10))


0    25.0
1    19.0
2    20.0
3    26.0
4    26.0
5    32.0
6    26.0
7    26.0
8    22.0
9    33.0
Name: edad, dtype: float64


In [17]:
# El año de nacimiento lo pasamos a entero
df['nacimiento'] = df['nacimiento'].astype('Int64')
print(df['nacimiento'].dtype)

Int64


In [18]:
# Damos un vistazo a nuestros datos limpios
print(df.dtypes)
print('\n')
print(df.isnull().sum())

ranking                              int64
jugador                             object
nacionalidad                        object
posicion                            object
equipo                              object
competicion                         object
edad                               float64
nacimiento                           Int64
partidos_jugados                     int64
partidos_titular                     int64
minutos_disputados                   int64
partidos_completos                 float64
goles                                int64
asistencias                          int64
goles_+_asistencias                  int64
goles_normales                       int64
goles_de_penal                       int64
penales_cobrados                     int64
tarjetas_amarillas                   int64
tarjetas_rojas                       int64
goles_por_partido                  float64
asistencias_por_partido            float64
goles_y_asistencias_por_partido    float64
goles_norma

In [19]:
# Guardamos lo trabajado hasta este momento
df.to_csv("datos/players_limpio.csv", index=False)
print("Archivo guardado correctamente")

Archivo guardado correctamente


### 1.6 Asignación de posición fija de jugadores


In [20]:
# Vemos cuales son los valores únicos de la columna de posiciones
print(df['posicion'].unique())
print(df['posicion'].value_counts())

['MF,FW' 'DF' 'FW,MF' 'MF' 'DF,MF' 'FW' 'GK' 'MF,DF' 'DF,FW']
posicion
MF       912
DF       680
FW       386
MF,FW    194
GK       171
FW,MF    142
DF,MF    133
MF,DF    104
DF,FW      2
Name: count, dtype: int64


In [21]:
# Si bien sabemos que en el fútbol los jugadores son polivalentes y pueden desempeñar su juego en multiples posiciones
# para efectos prácticos del proyecto asignaremos una sola posición la cual es la posición principal de cada jugador.
df['posicion'] = df['posicion'].str.split(',').str[0]

In [22]:
# traducimos el nombre de la posición
df['posicion'] = df['posicion'].replace({
    'GK': 'Porteros',
    'DF': 'Defensas',
    'MF': 'Mediocampistas',
    'FW': 'Delanteros'
})

print(df['posicion'].value_counts())

posicion
Mediocampistas    1210
Defensas           815
Delanteros         528
Porteros           171
Name: count, dtype: int64


In [23]:
# Guardamos lo trabajado
df.to_csv("datos/players_limpio.csv", index=False)
print("Archivo guardado correctamente")

Archivo guardado correctamente


### 1.7 Análisis de todo el dataset para calcular valores de referencia

In [24]:
# Ahora haremos un análisis de benchmarking antes de filtrar los jugadores españoles.
# El objetivo es calcular valores de referencia por posición usando todos los jugadores
# de las 5 grandes ligas, para luego poder contextualizar a los jugadores españoles dentro de ese universo.


In [25]:
# importamos el dataset limpio previamente
df = pd.read_csv("datos/players_limpio.csv")

In [26]:
# verificamos que todo esté en orden
print(df.shape)
print(df.head())

(2724, 25)
   ranking              jugador nacionalidad        posicion  \
0        1     Brenden Aaronson       us USA  Mediocampistas   
1        2          Zach Abbott      eng ENG        Defensas   
2        3  Jones El-Abdellaoui       ma MAR  Mediocampistas   
3        4        Himad Abdelli       dz ALG      Delanteros   
4        5        Himad Abdelli       dz ALG  Mediocampistas   

              equipo         competicion  edad  nacimiento  partidos_jugados  \
0       Leeds United  eng Premier League  25.0      2000.0                30   
1  Nottingham Forest  eng Premier League  19.0      2006.0                 2   
2         Celta Vigo          es La Liga  20.0      2006.0                19   
3          Marseille          fr Ligue 1  26.0      1999.0                 4   
4             Angers          fr Ligue 1  26.0      1999.0                13   

   partidos_titular  ...  goles_normales  goles_de_penal  penales_cobrados  \
0                24  ...               4     

In [27]:
# Queremos ver los valores unicos por posiciones, y cuantos jugadores hay en cada combinación de posición
print(df['posicion'].unique())
print('\n')
print(df['posicion'].value_counts())

['Mediocampistas' 'Defensas' 'Delanteros' 'Porteros']


posicion
Mediocampistas    1210
Defensas           815
Delanteros         528
Porteros           171
Name: count, dtype: int64


### 1.8 DECISIÓN DE NEGOCIO

In [28]:
# Como decisión de negocio de esta dataset solo nos servirá para filtrar los jugadores españoles con mas partidos jugados
# Por lo que eliminaremos columnas que no nos servirán para este análisis
# Si bien queria calcular los valores de referencia por posición de todas las 5 ligas, al final vamos escoger los mejores españoles para convocar
# por lo que no nos servirá de nada en este proyecto hacer una análisis de todos los jugadores de las 5 grandes ligas, por
# lo que nos concentraremos unicamente en escoger lo mejor de los jugadores españoles
print(df.columns.tolist())

['ranking', 'jugador', 'nacionalidad', 'posicion', 'equipo', 'competicion', 'edad', 'nacimiento', 'partidos_jugados', 'partidos_titular', 'minutos_disputados', 'partidos_completos', 'goles', 'asistencias', 'goles_+_asistencias', 'goles_normales', 'goles_de_penal', 'penales_cobrados', 'tarjetas_amarillas', 'tarjetas_rojas', 'goles_por_partido', 'asistencias_por_partido', 'goles_y_asistencias_por_partido', 'goles_normales_por_partido', 'goles_normales_y_asistencias']


In [29]:
# Nos quedaremos solo con las columnas que nos interesan para obtener la lista de jugadores españoles elegibles
df = df[['jugador', 'nacionalidad', 'posicion','partidos_jugados','partidos_titular','minutos_disputados','partidos_completos']]
print(df.shape)
print(df.head())

(2724, 7)
               jugador nacionalidad        posicion  partidos_jugados  \
0     Brenden Aaronson       us USA  Mediocampistas                30   
1          Zach Abbott      eng ENG        Defensas                 2   
2  Jones El-Abdellaoui       ma MAR  Mediocampistas                19   
3        Himad Abdelli       dz ALG      Delanteros                 4   
4        Himad Abdelli       dz ALG  Mediocampistas                13   

   partidos_titular  minutos_disputados  partidos_completos  
0                24                1975                21.9  
1                 1                 120                 1.3  
2                 4                 593                 6.6  
3                 0                  67                 0.7  
4                11                 943                10.5  


### 1.9 Filtrado solo de Jugadores Espanoles

In [30]:
# Escogemos de todo el dataset solo los jugadores españoles
df_spain = df[df['nacionalidad'] == 'es ESP']
print(df_spain.shape)
print(df_spain['jugador'].unique())

(0, 7)
[]


In [31]:
# Tengo problemas porque los datos arrojados no son los que esperaba
df_spain = df[df['nacionalidad'] == 'es ESP']
print(df_spain.shape)
print(df_spain['jugador'].unique())

(0, 7)
[]


In [32]:
# Consulto los 20 primeros valores unicos para ver como los está leyendo python en la columna de nacionalidad
print(df['nacionalidad'].unique()[:20])

['us\xa0USA' 'eng\xa0ENG' 'ma\xa0MAR' 'dz\xa0ALG' 'tn\xa0TUN' 'gh\xa0GHA'
 'sa\xa0KSA' 'il\xa0ISR' 'fr\xa0FRA' 'br\xa0BRA' 'ge\xa0GEO' 'mg\xa0MAD'
 'it\xa0ITA' 'de\xa0GER' 'ng\xa0NGA' 'sct\xa0SCO' 'at\xa0AUT' 'nl\xa0NED'
 'ci\xa0CIV' 'me\xa0MNE']


In [33]:
# Ya he descubierto que pasa, el espacio está escrito como \xa0, procedo a remplazarlo por espacio en blanco
df['nacionalidad'] = df['nacionalidad'].str.replace('\xa0', ' ')
print(df['nacionalidad'].unique()[:20])

['us USA' 'eng ENG' 'ma MAR' 'dz ALG' 'tn TUN' 'gh GHA' 'sa KSA' 'il ISR'
 'fr FRA' 'br BRA' 'ge GEO' 'mg MAD' 'it ITA' 'de GER' 'ng NGA' 'sct SCO'
 'at AUT' 'nl NED' 'ci CIV' 'me MNE']


In [34]:
# ahora si procedo a filtrar los jugadores españoles
df_spain = df[df['nacionalidad'] == 'es ESP']
print(df_spain.shape)
print(df_spain['jugador'].unique())

(392, 7)
['Julen Agirrezabala' 'Diego Aguado' 'Marc Aguado' 'Pablo Agudín'
 'Lucas Ahijado' 'Raúl Albiol' 'Carles Aleñá' 'Alfon' 'Marcos Alonso'
 'Adrià Altimira' 'Sergi Altimira' 'Carlos Álvarez' 'Hugo Álvarez'
 'Manuel Ángel' 'Miguel Ángel Rubio' 'Miguel Ángel Sierra' 'Angeliño'
 'Andrés Antañón' 'Antoniu' 'Jesús Areso' 'Iñigo Arguibide' 'Raúl Asencio'
 'Iago Aspas' 'Lander Astiazaran' 'César Azpilicueta' 'Alex Baena'
 'Alejandro Balde' 'Carlos Ballestero' 'Kike Barja' 'Ander Barrenetxea'
 'Pablo Barrios' 'Marc Bartra' 'Samu Becerra' 'Héctor Bellerín'
 'Fran Beltrán' 'Iker Benito' 'Yuri Berchiche' 'Álex Berenguer'
 'Adrian Bernabe' 'Marc Bernal' 'Pedro Bigas' 'Antonio Blanco'
 'Adama Boiro' 'Iker Bravo' 'Abel Bretones' 'Brugui' 'Hugo Bueno'
 'Manu Bueno' 'Jorge Cabello' 'Fernando Calero' 'Dani Calvo'
 'Sergio Camello' 'Sergi Cardona' 'Kevin Carlos' 'Carmona'
 'Sergio Carreira' 'Gorka Carrera' 'Álvaro Carreras' 'Dani Carvajal'
 'Marc Casado' 'Castrin' 'Jonny Castro' 'Catena' 'Santi Ca

### 1.10 Filtrado de Jugadores con mejor rendimiento en las temporadas 2024-2025 2025-2026

In [35]:
# Ahora que tenemos filtrados los jugadores españoles calcularemos las medias de participación 
media_partidos = df_spain['partidos_jugados'].mean()
media_titular = df_spain['partidos_titular'].mean()
media_minutos = df_spain['minutos_disputados'].mean()

print(f"Media partidos jugados: {media_partidos:.1f}")
print(f"Media partidos titular: {media_titular:.1f}")
print(f"Media minutos disputados: {media_minutos:.1f}")

Media partidos jugados: 15.9
Media partidos titular: 11.1
Media minutos disputados: 992.5


In [36]:
# Escogeremos solo los jugadores que estén por encima de la media de participación de partidos, es decir los jugadores con mas ritmo de competencia.
df_elegibles = df_spain[
    (df_spain['partidos_jugados'] > 15.9) &
    (df_spain['partidos_titular'] > 11.1) &
    (df_spain['minutos_disputados'] > 992.5)
]

print(df_elegibles.shape)
print(df_elegibles['jugador'].unique())

(169, 7)
['Julen Agirrezabala' 'Marc Aguado' 'Carles Aleñá' 'Marcos Alonso'
 'Carlos Álvarez' 'Jesús Areso' 'Raúl Asencio' 'Alex Baena'
 'Alejandro Balde' 'Ander Barrenetxea' 'Pablo Barrios' 'Marc Bartra'
 'Héctor Bellerín' 'Yuri Berchiche' 'Álex Berenguer' 'Adrian Bernabe'
 'Pedro Bigas' 'Antonio Blanco' 'Abel Bretones' 'Hugo Bueno'
 'Fernando Calero' 'Sergi Cardona' 'Carmona' 'Sergio Carreira'
 'Álvaro Carreras' 'Jonny Castro' 'Catena' 'Pep Chavarría' 'Víctor Chust'
 'Santi Comesaña' 'Copete' 'David Costas' 'Pau Cubarsí' 'Marc Cucurella'
 'Sergi Darder' 'Pedro Díaz' 'Pablo Durán' 'Hugo Duro' 'Aarón Escandell'
 'Edu Expósito' 'Aleix Febas' 'Kiko Femenía' 'Roberto Férnandez'
 'Pablo Fornals' 'Jorge de Frutos' 'Iñigo Ruiz de Galarreta'
 'Aleix García' 'Álvaro García' 'Eric García' 'Joan García' 'Rubén García'
 'David de Gea' 'Bryan Gil' 'Mario Gila' 'Sergio Gómez' 'Nicolás González'
 'Urko González' 'Jon Gorrotxategi' 'Álex Grimaldo' 'Javier Guerra'
 'Gorka Guruzeta' 'Mario Hermoso' 'Jo

In [37]:
# Guardaremos esta lista de 169 jugadores como un archivo csv para trabajarla fuera de python,
# ya que necesitaremos asignar manualmente la posición fuerte a cada jugador para posterior análisis individual.
# He decidido hacerlo en google sheets porque iba ser mas práctico y rápido para mi hacerlo alli, ya que a medida
# que iba consultando el nombre del jugador iba asignando la subposición inmediatamente por mi propio conocimiento
# o por una investigación rapida, también pude hacerlo aqui en python, pero la introducción de datos manuales en python 
# me parece mas lenta ya que tenia que también escribir en diccionario el nombre del jugador y la subposición.
df_elegibles.to_csv("datos/jugadores_elegibles.csv", index=False)
print("Archivo guardado correctamente")

Archivo guardado correctamente


# 2.0 Asignación de posición fuerte de cada jugador "subposición"

In [38]:
# Una vez que hemos agregado la subposición de cada jugador manualmente en google sheets procedemos a cargar
# el archivo guardado en formato .csv a python
df_elegibles = pd.read_csv("datos/jugadores_elegibles_con_posicion.csv")

print(df_elegibles.shape)
print(df_elegibles.columns.tolist())
print(df_elegibles.head())

(169, 8)
['jugador', 'nacionalidad', 'posicion', 'sub_posición', 'partidos_jugados', 'partidos_titular', 'minutos_disputados', 'partidos_completos']
              jugador nacionalidad        posicion            sub_posición  \
0  Julen Agirrezabala       es ESP        Porteros                 Portero   
1         Marc Aguado       es ESP  Mediocampistas  Medio_centro_defensivo   
2        Carles Aleñá       es ESP  Mediocampistas   Medio_centro_ofensivo   
3       Marcos Alonso       es ESP        Defensas         Defensa_central   
4      Carlos Álvarez       es ESP  Mediocampistas   Medio_centro_ofensivo   

   partidos_jugados  partidos_titular  minutos_disputados partidos_completos  
0                18                18                1620                 18  
1                28                23                1933               21,5  
2                29                22                1806               20,1  
3                24                23                2063         

In [39]:
# Inspeccionamos todas las subposiciones que agregamos
print(df_elegibles['sub_posición'].unique())

['Portero' 'Medio_centro_defensivo' 'Medio_centro_ofensivo'
 'Defensa_central' 'Lateral_derecho' 'Extremo_izquierdo'
 'Lateral_izquierdo' 'Delantero_centro' 'Extremo_derecho']


In [40]:
# Ahora queremos saber cuantos jugadores hay por posición para ver como está distribuida la lista de los 169 elegibles
# antes de hacer el corte final de 26
print(df_elegibles['sub_posición'].value_counts())

sub_posición
Defensa_central           34
Medio_centro_defensivo    32
Lateral_izquierdo         20
Delantero_centro          18
Lateral_derecho           17
Medio_centro_ofensivo     16
Portero                   14
Extremo_izquierdo         12
Extremo_derecho            6
Name: count, dtype: int64


In [41]:
# Guardamos lo trabajado hasta aqui para luego ir a trabajar los jugadores por posicion
df_elegibles.to_csv("datos/jugadores_elegibles_con_posicion.csv", index=False)
print("Archivo guardado correctamente")

Archivo guardado correctamente


# 3. Análisis, Asignación de métricas y construcción de la tabla de porteros.

### 3.1 Importación y limpieza de Datasets con las métricas de los porteros

In [42]:
# importamos lo datos de los jugadores elegibles por posición que ya hemos limpiado.
# importaremos también la base de datos especificos de los porteros ya que contienen métricas
# especificas para esta posicion y que hemos copiado desde fbref
df_elegibles = pd.read_csv("datos/jugadores_elegibles_con_posicion.csv")
df_porteros_raw = pd.read_csv("datos/fbref_porteros.csv")
print(df_elegibles.shape)
print(df_porteros_raw.shape)

(169, 8)
(173, 1)


In [43]:
# df_elegibles se encuentra bien, pero df_porteros solo me muestra una columna por lo que al parecer el separador no es ','
# probamos de otra manera
df_porteros_raw = pd.read_csv("datos/fbref_porteros.csv", sep=';', encoding='latin-1')

print(df_porteros_raw.shape)
print(df_porteros_raw.columns.tolist())
print(df_porteros_raw.head())

(173, 28)
['ï»¿Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', 'MP', 'Starts', 'Min', '90s', 'GA', 'GA90', 'SoTA', 'Saves', 'Save%', 'W', 'D', 'L', 'CS', 'CS%', 'PKatt', 'PKA', 'PKsv', 'PKm', 'Save%.1', 'Matches']
   ï»¿Rk              Player   Nation Pos            Squad  \
0      1  Julen Agirrezabala  esÂ ESP  GK         Valencia   
1      2             Alisson  brÂ BRA  GK        Liverpool   
2      3    NÃ­colas Andrade  brÂ BRA  GK             Pisa   
3      4     Alphonse Areola  frÂ FRA  GK  West Ham United   
4      5         Paul Argney  frÂ FRA  GK         Le Havre   

                  Comp     Age  Born  MP  Starts  ...  D   L  CS   CS%  PKatt  \
0          esÂ La Liga  25-111  2000  18      18  ...  7   8   4  22.2      4   
1  engÂ Premier League  33-196  1992  25      25  ...  7   5   8  32.0      2   
2          itÂ Serie A  38-004  1988   6       6  ...  0   5   0   0.0      1   
3  engÂ Premier League  33-048  1993  20      20  ...  5  11   0   0.0   

In [44]:
# Procedemos a limpiar el dataset de porteros y seleccionamos solo las columnas que necesitamos mas player y nation
# e incluiremos la columna 90s para calcular los goles salvados por partido.
df_porteros = df_porteros_raw[['Player', 'Nation', 'SoTA', 'GA', 'Saves', 'GA90', 'PKatt', 'PKA', 'PKsv', 'Save%.1', '90s']]

print(df_porteros.shape)
print(df_porteros.head())

(173, 11)
               Player   Nation  SoTA  GA  Saves  GA90  PKatt  PKA  PKsv  \
0  Julen Agirrezabala  esÂ ESP    79  30     52  1.67      4    3     1   
1             Alisson  brÂ BRA    84  30     56  1.20      2    2     0   
2    NÃ­colas Andrade  brÂ BRA    26  14     12  2.33      1    0     0   
3     Alphonse Areola  frÂ FRA   108  37     76  1.85      6    5     0   
4         Paul Argney  frÂ FRA     3   1      2  1.00      0    0     0   

   Save%.1   90s  
0     25.0  18.0  
1      0.0  25.0  
2      NaN   6.0  
3      0.0  20.0  
4      NaN   1.0  


In [45]:
# Limpiamos la columna Nation
df_porteros['Nation'] = df_porteros['Nation'].str.replace('Â', '')
print(df_porteros['Nation'].unique()[:10])

['es\xa0ESP' 'br\xa0BRA' 'fr\xa0FRA' 'de\xa0GER' 'id\xa0IDN' 'ar\xa0ARG'
 'tr\xa0TUR' 'fi\xa0FIN' 'nl\xa0NED' 'be\xa0BEL']


C:\Users\lgmar\AppData\Local\Temp\ipykernel_5700\849724458.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_porteros['Nation'] = df_porteros['Nation'].str.replace('Â', '')


In [46]:
# limpiamos ahora el \xa y usamos copy() segun lo recomendado por pandas

df_porteros = df_porteros_raw[['Player', 'Nation', 'SoTA', 'GA', 'Saves', 'GA90', 'PKatt', 'PKA', 'PKsv', 'Save%.1', '90s']].copy()

df_porteros['Nation'] = df_porteros['Nation'].str.replace('Â', '').str.replace('\xa0', ' ')

print(df_porteros['Nation'].unique()[:10])

['es ESP' 'br BRA' 'fr FRA' 'de GER' 'id IDN' 'ar ARG' 'tr TUR' 'fi FIN'
 'nl NED' 'be BEL']


In [47]:
# Ahora solo queremos ver los porteros Españoles
df_porteros_esp = df_porteros[df_porteros['Nation'] == 'es ESP']

print(df_porteros_esp.shape)
print(df_porteros_esp['Player'].unique())


(19, 11)
['Julen Agirrezabala' 'Dani CÃ¡rdenas' 'Pablo CuÃ±at' 'AarÃ³n Escandell'
 'Joan GarcÃ\xada' 'David de Gea' 'Sergio Herrera' 'Pau LÃ³pez'
 'Josep Martinez' 'IÃ±aki PeÃ±a' 'David Raya' 'Ã\x81lex Remiro'
 'Leo RomÃ¡n' 'Robert SÃ¡nchez' 'Unai SimÃ³n' 'Antonio Sivera'
 'David Soria' 'Arnau Tenas' 'Ã\x81lvaro VallÃ©s']


In [48]:
# Tenemos 19 porteros, de estos 19 solo 3 serán los elegidos para el mundial, pero antes debemos eliminar los caracteres
# especiales que tenemos en los nombres por acentos y ñs volviendo a importar los datos con la codificación correcta
df_porteros_raw = pd.read_csv(
    "datos/fbref_porteros.csv",
    sep=';',
    encoding='utf-8-sig'
)

print(df_porteros_raw['Player'].head())

0    Julen Agirrezabala
1               Alisson
2       Nícolas Andrade
3       Alphonse Areola
4           Paul Argney
Name: Player, dtype: object


In [49]:
# repetimos los pasos de limpieza otra vez
# Seleccionar columnas necesarias
df_porteros = df_porteros_raw[['Player', 'Nation', 'SoTA', 'GA', 'Saves', 'GA90', 'PKatt', 'PKA', 'PKsv', 'Save%.1', '90s']].copy()

# Limpiar columna Nation
df_porteros['Nation'] = df_porteros['Nation'].str.replace('Â', '').str.replace('\xa0', ' ')

# Filtrar porteros españoles
df_porteros_esp = df_porteros[df_porteros['Nation'] == 'es ESP']

print(df_porteros_esp.shape)
print(df_porteros_esp['Player'].unique())

(19, 11)
['Julen Agirrezabala' 'Dani Cárdenas' 'Pablo Cuñat' 'Aarón Escandell'
 'Joan García' 'David de Gea' 'Sergio Herrera' 'Pau López'
 'Josep Martinez' 'Iñaki Peña' 'David Raya' 'Álex Remiro' 'Leo Román'
 'Robert Sánchez' 'Unai Simón' 'Antonio Sivera' 'David Soria'
 'Arnau Tenas' 'Álvaro Vallés']


### 3.2 Cálculo de métricas importantes de los Porteros

In [50]:
# Ahora si tenemos todo bien, los 19 porteros con los nombres correctos 
# Ahora calcularemos la métrica de goles salvados por partido
# Calculamos goles salvados por partido
df_porteros_esp = df_porteros_esp.copy()
df_porteros_esp['salvadas_por_partido'] = (df_porteros_esp['Saves'] / df_porteros_esp['90s']).round(2)

# Renombramos columnas al español
df_porteros_esp = df_porteros_esp.rename(columns={
    'Player': 'jugador',
    'Nation': 'nacionalidad',
    'SoTA': 'disparos_a_puerta',
    'GA': 'goles_en_contra',
    'Saves': 'goles_salvados',
    'GA90': 'goles_en_contra_por_partido',
    'PKatt': 'intentos_penalty',
    'PKA': 'penaltys_permitidos',
    'PKsv': 'penaltys_salvados',
    'Save%.1': 'porcentaje_penaltys_salvados'
})

# Eliminamos la columna 90s que ya no necesitamos
df_porteros_esp = df_porteros_esp.drop(columns=['90s'])

print(df_porteros_esp.columns.tolist())
print(df_porteros_esp.head())

['jugador', 'nacionalidad', 'disparos_a_puerta', 'goles_en_contra', 'goles_salvados', 'goles_en_contra_por_partido', 'intentos_penalty', 'penaltys_permitidos', 'penaltys_salvados', 'porcentaje_penaltys_salvados', 'salvadas_por_partido']
               jugador nacionalidad  disparos_a_puerta  goles_en_contra  \
0   Julen Agirrezabala       es ESP                 79               30   
20       Dani Cárdenas       es ESP                  4                3   
27         Pablo Cuñat       es ESP                 16                5   
43     Aarón Escandell       es ESP                172               48   
48         Joan García       es ESP                 83               19   

    goles_salvados  goles_en_contra_por_partido  intentos_penalty  \
0               52                         1.67                 4   
20               1                         3.00                 0   
27              11                         2.50                 0   
43             130                  

### 3.3 Filtración de mejores porteros elegibles.

In [51]:
# Ahora necesitaremos unir ambos datasets para quedarnos solo con los porteros que pasaron el filtro de participación

df_porteros_final = df_elegibles[df_elegibles['sub_posición'] == 'Portero'].merge(
    df_porteros_esp,
    on='jugador',
    how='inner'
)

print(df_porteros_final.shape)
print(df_porteros_final['jugador'].unique())

(14, 18)
['Julen Agirrezabala' 'Aarón Escandell' 'Joan García' 'David de Gea'
 'Sergio Herrera' 'Iñaki Peña' 'David Raya' 'Álex Remiro' 'Leo Román'
 'Robert Sánchez' 'Unai Simón' 'Antonio Sivera' 'David Soria'
 'Álvaro Vallés']


In [52]:
# Ahora visualizamos los 14 porteros elegibles con todas sus métricas
print(df_porteros_final[['jugador', 'partidos_jugados', 'partidos_titular', 'minutos_disputados', 'disparos_a_puerta', 'goles_en_contra', 'goles_salvados', 'goles_en_contra_por_partido', 'salvadas_por_partido']].sort_values('goles_en_contra_por_partido').to_string())

               jugador  partidos_jugados  partidos_titular  minutos_disputados  disparos_a_puerta  goles_en_contra  goles_salvados  goles_en_contra_por_partido  salvadas_por_partido
6           David Raya                31                31                2790                 75               24              50                         0.75                  1.56
2          Joan García                23                23                2070                 83               19              66                         0.76                  2.64
12         David Soria                29                29                2610                124               32              94                         1.03                  3.03
4       Sergio Herrera                29                29                2610                133               38             103                         1.23                  3.32
13       Álvaro Vallés                20                20                1800            

In [53]:
# Aunque a partir de este punto ya podriamos tomar una decisión de los mejores 3 porteros para la convocatoria,
# no es la mejor forma de hacerlo, y veremos los datos graficamente usando powerbi, de momento solo guardaremos 
# el trabajo de limpieza y preparación de los datos hecho hasta ahora
df_porteros_final.to_csv("datos/porteros_final.csv", index=False)
print("Archivo guardado correctamente")

Archivo guardado correctamente


# 4. Análisis, Asignación de métricas y construcción de tablas de laterales.

In [54]:
# importamos la libreria pandas para empezar ahora a trabajar con los laterales, da igual si son laterales izquierdos
# o derechos, ambos van a ser evaluados con las mismas métricas
import pandas as pd

# Jugadores elegibles con sub-posición
df_elegibles = pd.read_csv("datos/jugadores_elegibles_con_posicion.csv")

print(df_elegibles.shape)
print(df_elegibles['sub_posición'].value_counts())

(169, 8)
sub_posición
Defensa_central           34
Medio_centro_defensivo    32
Lateral_izquierdo         20
Delantero_centro          18
Lateral_derecho           17
Medio_centro_ofensivo     16
Portero                   14
Extremo_izquierdo         12
Extremo_derecho            6
Name: count, dtype: int64


### 4.1 Importación y limpieza de un Dataset con métricas claves para los laterales.

In [55]:
# importamos un dataset de jugadores europeos defensivos, ya que en el dataset original no nos aparecen las métricas
# especificas que esperamos encontrar en lo laterales
df_defensiva_raw = pd.read_csv(
    "datos/fbref_defensiva.csv",
    sep=';',
    encoding='utf-8-sig'
)

print(df_defensiva_raw.shape)
print(df_defensiva_raw.columns.tolist())

(2742, 26)
['Rk', 'Player', 'Nation', 'Pos', 'Squad', 'Comp', 'Age', 'Born', '90s', 'Tkl', 'TklW', 'Def 3rd', 'Mid 3rd', 'Att 3rd', 'Tkl.1', 'Att', 'Tkl%', 'Lost', 'Blocks', 'Sh', 'Pass', 'Int', 'Tkl+Int', 'Clr', 'Err', 'Matches']


In [56]:
# Seleccionaremos solo las columnas que necesitamos y limpiaremos la columna de nacionalidad
df_defensiva = df_defensiva_raw[['Player', 'Nation', 'TklW', 'Int']].copy()

df_defensiva['Nation'] = df_defensiva['Nation'].str.replace('Â', '').str.replace('\xa0', ' ')

print(df_defensiva.head())
print(df_defensiva.shape)

                Player   Nation  TklW  Int
0     Brenden Aaronson   us USA    22   13
1          Zach Abbott  eng ENG     2    2
2  Jones El-Abdellaoui   ma MAR     3    3
3        Himad Abdelli   dz ALG     1    1
4        Himad Abdelli   dz ALG    17   13
(2742, 4)


In [57]:
# Ahora necesitamos cruzar 3 fuentes de datos df_elegibles.csv players_limpio.csv y df_defensiva que es la que acabamos de limpiar
# primero cargaremos players_limpio.csv y agregaremos las columnas de jugadores, goles y asistencias
df_players = pd.read_csv("datos/players_limpio.csv")

df_players = df_players[['jugador', 'goles', 'asistencias']].copy()

print(df_players.head())

               jugador  goles  asistencias
0     Brenden Aaronson      4            3
1          Zach Abbott      0            0
2  Jones El-Abdellaoui      2            0
3        Himad Abdelli      0            0
4        Himad Abdelli      2            0


### 4.2 Unión de Datasets para seleccionar la métricas claves de los laterales derechos.

In [58]:
# Filtramos laterales derechos elegibles
df_lat_der = df_elegibles[df_elegibles['sub_posición'] == 'Lateral_derecho'].copy()

# Merge con goles y asistencias
df_lat_der = df_lat_der.merge(df_players, on='jugador', how='left')

# Merge con datos defensivos
df_defensiva_renamed = df_defensiva.rename(columns={'Player': 'jugador'})
df_lat_der = df_lat_der.merge(df_defensiva_renamed[['jugador', 'TklW', 'Int']], on='jugador', how='left')

# Renombrar columnas defensivas
df_lat_der = df_lat_der.rename(columns={
    'TklW': 'duelos_ganados',
    'Int': 'intercepciones'
})

print(df_lat_der.shape)
print(df_lat_der[['jugador', 'partidos_jugados', 'partidos_titular', 'minutos_disputados', 'goles', 'asistencias', 'duelos_ganados', 'intercepciones']])

(23, 12)
                  jugador  partidos_jugados  partidos_titular  \
0             Jesús Areso                23                13   
1         Héctor Bellerín                18                13   
2      José Angel Carmona                27                22   
3         Sergio Carreira                25                21   
4            Kiko Femenía                24                22   
5   Juan Antonio Iglesias                28                27   
6            Álex Jiménez                27                22   
7            Álex Jiménez                27                22   
8            Álex Jiménez                27                22   
9            Álex Jiménez                27                22   
10        Marcos Llorente                25                21   
11           Pablo Maffeo                23                22   
12         Arnau Martinez                26                24   
13         Óscar Mingueza                25                18   
14            Pa

### 4.3 Limpieza de duplicados

In [59]:
# Observamos varios duplicados para Álex Jiménez y Álvaro Núñez, eso es porque estuvieron en mas de un equipo,
# asi que agruparemos cada jugador y sumaremos sus estadisticas
df_defensiva_grouped = df_defensiva_renamed.groupby('jugador').agg({
    'TklW': 'sum',
    'Int': 'sum'
}).reset_index()

# Rehacemos el merge con los datos agrupados
df_lat_der = df_elegibles[df_elegibles['sub_posición'] == 'Lateral_derecho'].copy()
df_lat_der = df_lat_der.merge(df_players, on='jugador', how='left')
df_lat_der = df_lat_der.merge(df_defensiva_grouped, on='jugador', how='left')

df_lat_der = df_lat_der.rename(columns={
    'TklW': 'duelos_ganados',
    'Int': 'intercepciones'
})

print(df_lat_der.shape)
print(df_lat_der[['jugador', 'partidos_jugados', 'goles', 'asistencias', 'duelos_ganados', 'intercepciones']])

(19, 12)
                  jugador  partidos_jugados  goles  asistencias  \
0             Jesús Areso                23    0.0          1.0   
1         Héctor Bellerín                18    1.0          2.0   
2      José Angel Carmona                27    NaN          NaN   
3         Sergio Carreira                25    2.0          1.0   
4            Kiko Femenía                24    1.0          1.0   
5   Juan Antonio Iglesias                28    NaN          NaN   
6            Álex Jiménez                27    1.0          0.0   
7            Álex Jiménez                27    0.0          0.0   
8         Marcos Llorente                25    0.0          4.0   
9            Pablo Maffeo                23    1.0          0.0   
10         Arnau Martinez                26    1.0          1.0   
11         Óscar Mingueza                25    1.0          4.0   
12            Pau Navarro                17    0.0          0.0   
13           Álvaro Núñez                19    0.0   

In [60]:
# persiste el problema de duplicados de Alex Jimenez y Alvaro Nuñez el cual está en df_elegibles que los tiene dos veces, no en la tabla defensiva.
# Carmona e iglesias sus nombres no coinciden exactamente entre datasets y por eso no aparecen sus valores
# Queremos ver por qué están duplicados
print(df_elegibles[df_elegibles['jugador'] == 'Álex Jiménez'])
print(df_elegibles[df_elegibles['jugador'] == 'Álvaro Núñez'])

         jugador nacionalidad  posicion     sub_posición  partidos_jugados  \
69  Álex Jiménez       es ESP  Defensas  Lateral_derecho                27   

    partidos_titular  minutos_disputados partidos_completos  
69                22                1958               21,8  
          jugador nacionalidad  posicion     sub_posición  partidos_jugados  \
108  Álvaro Núñez       es ESP  Defensas  Lateral_derecho                19   

     partidos_titular  minutos_disputados partidos_completos  
108                19                1647               18,3  


In [61]:
# en df_elegibles solo aparecen una vez cada uno. al parecer el problema de duplicados viene del merge con df_players. 
# en este momento me siento bastante estancado =S
# sigo mirando de donde viene el problema
print(df_players[df_players['jugador'] == 'Álex Jiménez'])
print(df_players[df_players['jugador'] == 'Álvaro Núñez'])

           jugador  goles  asistencias
1202  Álex Jiménez      1            0
1203  Álex Jiménez      0            0
           jugador  goles  asistencias
1858  Álvaro Núñez      0            2
1859  Álvaro Núñez      0            0


In [62]:
# El problema es que estos jugadores aparecen dos veces en players_limpio.csv porque jugaron en dos 
# equipos distintos durante la temporada. FBref registra una fila por cada equipo.
# La solución es agrupar df_players por jugador sumando goles y asistencias antes de hacer el merge
df_players_grouped = df_players.groupby('jugador').agg({
    'goles': 'sum',
    'asistencias': 'sum'}).reset_index()

# Rehacemos todo el merge
df_lat_der = df_elegibles[df_elegibles['sub_posición'] == 'Lateral_derecho'].copy()
df_lat_der = df_lat_der.merge(df_players_grouped, on='jugador', how='left')
df_lat_der = df_lat_der.merge(df_defensiva_grouped, on='jugador', how='left')

df_lat_der = df_lat_der.rename(columns={
    'TklW': 'duelos_ganados',
    'Int': 'intercepciones'})

print(df_lat_der.shape)
print(df_lat_der[['jugador', 'partidos_jugados', 'goles', 'asistencias', 'duelos_ganados', 'intercepciones']])

(17, 12)
                  jugador  partidos_jugados  goles  asistencias  \
0             Jesús Areso                23    0.0          1.0   
1         Héctor Bellerín                18    1.0          2.0   
2      José Angel Carmona                27    NaN          NaN   
3         Sergio Carreira                25    2.0          1.0   
4            Kiko Femenía                24    1.0          1.0   
5   Juan Antonio Iglesias                28    NaN          NaN   
6            Álex Jiménez                27    1.0          0.0   
7         Marcos Llorente                25    0.0          4.0   
8            Pablo Maffeo                23    1.0          0.0   
9          Arnau Martinez                26    1.0          1.0   
10         Óscar Mingueza                25    1.0          4.0   
11            Pau Navarro                17    0.0          0.0   
12           Álvaro Núñez                19    0.0          2.0   
13            Pedro Porro                27    0.0   

In [63]:
# Perfecto, ahora sí tenemos los 17 laterales derechos con todas las métricas y sin NaN.
# Guardamo este dataframe como CSV:
df_lat_der.to_csv("datos/laterales_derechos_final.csv", index=False)
print("Archivo guardado correctamente")

Archivo guardado correctamente


### 4.4 Unión de Datasets y limpieza de datos para seleccionar la métricas claves de los laterales izquierdos

In [64]:
# hacemos el mismo procedimiento pero ahora con los laterales izquierdos
df_lat_izq = df_elegibles[df_elegibles['sub_posición'] == 'Lateral_izquierdo'].copy()
df_lat_izq = df_lat_izq.merge(df_players_grouped, on='jugador', how='left')
df_lat_izq = df_lat_izq.merge(df_defensiva_grouped, on='jugador', how='left')

df_lat_izq = df_lat_izq.rename(columns={
    'TklW': 'duelos_ganados',
    'Int': 'intercepciones'
})

print(df_lat_izq.shape)
print(df_lat_izq[['jugador', 'partidos_jugados', 'goles', 'asistencias', 'duelos_ganados', 'intercepciones']])

(20, 12)
            jugador  partidos_jugados  goles  asistencias  duelos_ganados  \
0   Alejandro Balde                21      0            2              18   
1    Yuri Berchiche                25      0            3              23   
2     Abel Bretones                22      1            0              11   
3        Hugo Bueno                28      1            1              30   
4     Sergi Cardona                23      0            3              27   
5   Álvaro Carreras                24      2            2              17   
6     Pep Chavarría                25      1            2              26   
7    Marc Cucurella                27      1            3              34   
8      Sergio Gómez                27      0            2              46   
9     Álex Grimaldo                22      6            6              20   
10       Javi López                17      0            1              20   
11   José Luis Gayà                25      1            1          

In [65]:
# Guardamos el dataframe para los laterales izquierdos.
df_lat_izq.to_csv("datos/laterales_izquierdos_final.csv", index=False)
print("Archivo guardado correctamente")

Archivo guardado correctamente


# 5. Análisis, Asignación de métricas y construcción de tabla de los defensas centrales

### 5.1 Importación de Datasets con las métricas claves de los defensas centrales.

In [66]:
# Ahora vamos con el Análisis de los defensas centrales
# Importamos la libreria de pandas y cargamos los datasets necesarios
import pandas as pd

# Jugadores elegibles con sub-posición
df_elegibles = pd.read_csv("datos/jugadores_elegibles_con_posicion.csv")

# Goles y asistencias
df_players = pd.read_csv("datos/players_limpio.csv")

# Tabla defensiva
df_defensiva_raw = pd.read_csv(
    "datos/fbref_defensiva.csv",
    sep=';',
    encoding='utf-8-sig'
)

print(df_elegibles.shape)
print(df_players.shape)
print(df_defensiva_raw.shape)

(169, 8)
(2724, 25)
(2742, 26)


### 5.2 Limpieza y unión de los datasets

In [67]:
# Para los defensas centrales tomaremos las siguientes métricas 
# 1. Partidos Jugados.
# 2. Partidos titular.
# 3. Minutos disputados.
# 4. Duelos ganados.
# 5. Intercepciones.

# Profesionalmente y personalmente me gustaria poder medir muchas mas métricas como por ejemplo duelos aereos ofensivos y defensivos,
# tiros bloqueados, pases entre lineas, sin embargo el acceso a los datos es muy limitado y por temas de practicidad no lo haremos
# con tanta profundidad.
# Agrupamos players por jugador

df_players_grouped = df_players.groupby('jugador').agg({
    'goles': 'sum',
    'asistencias': 'sum'
}).reset_index()

# Limpiamos la tabla defensiva
df_defensiva = df_defensiva_raw[['Player', 'Nation', 'TklW', 'Int']].copy()
df_defensiva['Nation'] = df_defensiva['Nation'].str.replace('Â', '').str.replace('\xa0', ' ')
df_defensiva_renamed = df_defensiva.rename(columns={'Player': 'jugador'})
df_defensiva_grouped = df_defensiva_renamed.groupby('jugador').agg({
    'TklW': 'sum',
    'Int': 'sum'
}).reset_index()

# Corregimos los nombres
df_elegibles['jugador'] = df_elegibles['jugador'].replace({
    'José Angel Carmona': 'Carmona',
    'Juan Antonio Iglesias': 'Iglesias'
})

# Filtramos defensas centrales y hacemos merges
df_def_cen = df_elegibles[df_elegibles['sub_posición'] == 'Defensa_central'].copy()
df_def_cen = df_def_cen.merge(df_defensiva_grouped, on='jugador', how='left')
df_def_cen = df_def_cen.rename(columns={
    'TklW': 'duelos_ganados',
    'Int': 'intercepciones'
})

print(df_def_cen.shape)
print(df_def_cen[['jugador', 'partidos_jugados', 'partidos_titular', 'minutos_disputados', 'duelos_ganados', 'intercepciones']])

(34, 10)
                jugador  partidos_jugados  partidos_titular  \
0         Marcos Alonso                24                23   
1          Raúl Asencio                19                14   
2           Marc Bartra                22                18   
3           Pedro Bigas                21                19   
4       Fernando Calero                21                18   
5          Jonny Castro                28                28   
6      Alejandro Catena                27                27   
7          Víctor Chust                23                19   
8           José Copete                21                19   
9          David Costas                19                18   
10          Pau Cubarsí                24                24   
11          Eric García                27                25   
12           Mario Gila                27                27   
13        Mario Hermoso                19                17   
14       Jorge Herrando                24     

In [68]:
# Queremos ver porque Catena y Copete tienen NaN
print(df_defensiva_grouped[df_defensiva_grouped['jugador'].str.contains('Catena')])
print(df_defensiva_grouped[df_defensiva_grouped['jugador'].str.contains('Copete')])

    jugador  TklW  Int
417  Catena    24   30
    jugador  TklW  Int
475  Copete    15   25


In [69]:
# df_elegibles tiene nombres completos pero en la tabla defensiva tienen nombres cortos, corregimos esto

df_elegibles['jugador'] = df_elegibles['jugador'].replace({
    'Alejandro Catena': 'Catena',
    'José Copete': 'Copete'
})

# toca Rehacer el filtro y merge
df_def_cen = df_elegibles[df_elegibles['sub_posición'] == 'Defensa_central'].copy()
df_def_cen = df_def_cen.merge(df_defensiva_grouped, on='jugador', how='left')
df_def_cen = df_def_cen.rename(columns={
    'TklW': 'duelos_ganados',
    'Int': 'intercepciones'})

print(df_def_cen[['jugador', 'partidos_jugados', 'partidos_titular', 'minutos_disputados', 'duelos_ganados', 'intercepciones']])

                jugador  partidos_jugados  partidos_titular  \
0         Marcos Alonso                24                23   
1          Raúl Asencio                19                14   
2           Marc Bartra                22                18   
3           Pedro Bigas                21                19   
4       Fernando Calero                21                18   
5          Jonny Castro                28                28   
6                Catena                27                27   
7          Víctor Chust                23                19   
8                Copete                21                19   
9          David Costas                19                18   
10          Pau Cubarsí                24                24   
11          Eric García                27                25   
12           Mario Gila                27                27   
13        Mario Hermoso                19                17   
14       Jorge Herrando                24              

In [70]:
# Tenemos 34 defensas centrales con todas sus métricas completas, guardamos.
df_def_cen.to_csv("datos/defensas_centrales_final.csv", index=False)
print("Archivo guardado correctamente")

Archivo guardado correctamente


# 6. Análisis, asignación de métricas y construcción de tablas de mediocampistas

### 6.1 Importación de los datasets con las métricas clave de los mediocampistas

In [71]:
# Vamos ahora hacer el análisis de los mediocampistas integrando nuevas bases de datos que me den sus métricas
# importamos pandas y cargamos los datos
import pandas as pd

# Jugadores elegibles con sub-posición
df_elegibles = pd.read_csv("datos/jugadores_elegibles_con_posicion.csv")

# Goles y asistencias
df_players = pd.read_csv("datos/players_limpio.csv")

# Tabla defensiva
df_defensiva_raw = pd.read_csv(
    "datos/fbref_defensiva.csv",
    sep=';',
    encoding='utf-8-sig'
)

print(df_elegibles.shape)
print(df_players.shape)
print(df_defensiva_raw.shape)

(169, 8)
(2724, 25)
(2742, 26)


### 6.2 Limpieza y unión de los Datasets

In [72]:
# Para los mediocampistas tanto defensivos y ofensivos usaremos las mismas métricas que usamos para los laterales
# Dejamos documentado que nos gustaria cubrir muchas mas métricas que permitan ver en detalle el rendimiento de 
# los jugadores, pero no tenemos acceso a todos los datos, por lo que lo haremos con las métricas básicas y mas 
# importantes que cubren un 85% de importancia en la decisión de convocatoria.

# Agrupamos players por jugador
df_players_grouped = df_players.groupby('jugador').agg({
    'goles': 'sum',
    'asistencias': 'sum'}).reset_index()

# Limpiamos la tabla defensiva
df_defensiva = df_defensiva_raw[['Player', 'Nation', 'TklW', 'Int']].copy()
df_defensiva['Nation'] = df_defensiva['Nation'].str.replace('Â', '').str.replace('\xa0', ' ')
df_defensiva_renamed = df_defensiva.rename(columns={'Player': 'jugador'})
df_defensiva_grouped = df_defensiva_renamed.groupby('jugador').agg({
    'TklW': 'sum',
    'Int': 'sum'
}).reset_index()

# Corregimos nombres conocidos
df_elegibles['jugador'] = df_elegibles['jugador'].replace({
    'José Angel Carmona': 'Carmona',
    'Juan Antonio Iglesias': 'Iglesias',
    'Alejandro Catena': 'Catena',
    'José Copete': 'Copete'
})

# Filtramos mediocampistas y hacemos los merges
df_meds = df_elegibles[df_elegibles['sub_posición'].isin(['Medio_centro_defensivo', 'Medio_centro_ofensivo'])].copy()
df_meds = df_meds.merge(df_players_grouped, on='jugador', how='left')
df_meds = df_meds.merge(df_defensiva_grouped, on='jugador', how='left')

df_meds = df_meds.rename(columns={
    'TklW': 'duelos_ganados',
    'Int': 'intercepciones'
})

print(df_meds.shape)
print(df_meds[['jugador', 'sub_posición', 'partidos_jugados', 'goles', 'asistencias', 'duelos_ganados', 'intercepciones']])

(48, 12)
                    jugador            sub_posición  partidos_jugados  goles  \
0               Marc Aguado  Medio_centro_defensivo                28      1   
1              Carles Aleñá   Medio_centro_ofensivo                29      0   
2            Carlos Álvarez   Medio_centro_ofensivo                26      3   
3             Pablo Barrios  Medio_centro_defensivo                21      1   
4            Adrian Bernabe  Medio_centro_defensivo                27      3   
5            Antonio Blanco  Medio_centro_defensivo                27      1   
6            Santi Comesaña  Medio_centro_defensivo                27      3   
7              Sergi Darder  Medio_centro_defensivo                29      1   
8                Pedro Díaz  Medio_centro_defensivo                26      0   
9              Edu Expósito  Medio_centro_defensivo                27      1   
10              Aleix Febas   Medio_centro_ofensivo                28      2   
11            Pablo Fornals   M

In [73]:
# Tenemos 48 mediocampistas con todas las métricas completas, procedemos a guardar.
df_meds.to_csv("datos/mediocampistas_final.csv", index=False)
print("Archivo guardado correctamente")

Archivo guardado correctamente


# 7. Análisis, asignación de métricas y construcción de tablas de los extremos.

### 7.1 Importación de los datasets con las métricas claves de los extremos

In [74]:
# Vamos ahora con el análisis de los extremos, importando bases de datos que nos den sus métricas mas importantes.
# importamos la libreria de pandas y cargamos los datos
import pandas as pd

# Jugadores elegibles con sub-posición
df_elegibles = pd.read_csv("datos/jugadores_elegibles_con_posicion.csv")

# Goles y asistencias
df_players = pd.read_csv("datos/players_limpio.csv")

# Tabla defensiva
df_defensiva_raw = pd.read_csv(
    "datos/fbref_defensiva.csv",
    sep=';',
    encoding='utf-8-sig'
)

print(df_elegibles.shape)
print(df_players.shape)
print(df_defensiva_raw.shape)

(169, 8)
(2724, 25)
(2742, 26)


### 7.2 Limpieza y unión de los Datasets.

In [75]:
# Usaremos las mismas métricas que para los mediocampistas ya que aunque no sea tan detallada
# abordan bastante información para la toma de decisión final de convocatoria
# Agrupamos players por jugador
df_players_grouped = df_players.groupby('jugador').agg({
    'goles': 'sum',
    'asistencias': 'sum'
}).reset_index()

# Limpiar tabla defensiva
df_defensiva = df_defensiva_raw[['Player', 'Nation', 'TklW', 'Int']].copy()
df_defensiva['Nation'] = df_defensiva['Nation'].str.replace('Â', '').str.replace('\xa0', ' ')
df_defensiva_renamed = df_defensiva.rename(columns={'Player': 'jugador'})
df_defensiva_grouped = df_defensiva_renamed.groupby('jugador').agg({
    'TklW': 'sum',
    'Int': 'sum'
}).reset_index()

# Corregimos nombres conocidos
df_elegibles['jugador'] = df_elegibles['jugador'].replace({
    'José Angel Carmona': 'Carmona',
    'Juan Antonio Iglesias': 'Iglesias',
    'Alejandro Catena': 'Catena',
    'José Copete': 'Copete'
})

# Filtramos extremos y hacemos merges
df_extremos = df_elegibles[df_elegibles['sub_posición'].isin(['Extremo_izquierdo', 'Extremo_derecho'])].copy()
df_extremos = df_extremos.merge(df_players_grouped, on='jugador', how='left')
df_extremos = df_extremos.merge(df_defensiva_grouped, on='jugador', how='left')

df_extremos = df_extremos.rename(columns={
    'TklW': 'duelos_ganados',
    'Int': 'intercepciones'
})

print(df_extremos.shape)
print(df_extremos[['jugador', 'sub_posición', 'partidos_jugados', 'goles', 'asistencias', 'duelos_ganados', 'intercepciones']])

(18, 12)
              jugador       sub_posición  partidos_jugados  goles  \
0          Alex Baena  Extremo_izquierdo                20      2   
1   Ander Barrenetxea  Extremo_izquierdo                23      3   
2      Álex Berenguer  Extremo_izquierdo                24      2   
3       Álvaro García  Extremo_izquierdo                29      4   
4        Rubén García    Extremo_derecho                26      2   
5           Bryan Gil  Extremo_izquierdo                23      0   
6         Diego López    Extremo_derecho                26      3   
7          Pere Milla  Extremo_izquierdo                26      6   
8     Alberto Moleiro  Extremo_izquierdo                28      9   
9        Víctor Muñoz  Extremo_izquierdo                29      5   
10        Yeremi Pino    Extremo_derecho                27      2   
11         Luis Rioja    Extremo_derecho                28      2   
12    Jesus Rodríguez  Extremo_izquierdo                26      1   
13       Aitor Ruibal    

In [76]:
# Tenemos 18 extremos con todas las métricas completas y guardamos
df_extremos.to_csv("datos/extremos_final.csv", index=False)
print("Archivo guardado correctamente")

Archivo guardado correctamente


# 8. Análisis, asignación de métricas y contrucción de tablas de los delanteros

### 8.1 Importación y limpieza de los datasets con las métricas claves de los delanteros.

In [77]:
# Hacemos el mismo procedimiento de obtención y limpieza de datos que hemos venido haciendo por posición pero
# ahora con los delanteros
import pandas as pd

# Cargar datos
df_elegibles = pd.read_csv("datos/jugadores_elegibles_con_posicion.csv")
df_players = pd.read_csv("datos/players_limpio.csv")
df_defensiva_raw = pd.read_csv(
    "datos/fbref_defensiva.csv",
    sep=';',
    encoding='utf-8-sig'
)

# Agrupar players por jugador
df_players_grouped = df_players.groupby('jugador').agg({
    'goles': 'sum',
    'asistencias': 'sum'
}).reset_index()

# Limpiar tabla defensiva
df_defensiva = df_defensiva_raw[['Player', 'Nation', 'TklW', 'Int']].copy()
df_defensiva['Nation'] = df_defensiva['Nation'].str.replace('Â', '').str.replace('\xa0', ' ')
df_defensiva_renamed = df_defensiva.rename(columns={'Player': 'jugador'})
df_defensiva_grouped = df_defensiva_renamed.groupby('jugador').agg({
    'TklW': 'sum',
    'Int': 'sum'
}).reset_index()

# Corregir nombres conocidos
df_elegibles['jugador'] = df_elegibles['jugador'].replace({
    'José Angel Carmona': 'Carmona',
    'Juan Antonio Iglesias': 'Iglesias',
    'Alejandro Catena': 'Catena',
    'José Copete': 'Copete'
})

# Filtrar delanteros y hacer merges
df_delanteros = df_elegibles[df_elegibles['sub_posición'] == 'Delantero_centro'].copy()
df_delanteros = df_delanteros.merge(df_players_grouped, on='jugador', how='left')
df_delanteros = df_delanteros.merge(df_defensiva_grouped, on='jugador', how='left')

df_delanteros = df_delanteros.rename(columns={
    'TklW': 'duelos_ganados',
    'Int': 'intercepciones'
})

print(df_delanteros.shape)
print(df_delanteros[['jugador', 'partidos_jugados', 'goles', 'asistencias', 'duelos_ganados', 'intercepciones']])

(18, 12)
                  jugador  partidos_jugados  goles  asistencias  \
0             Pablo Durán                22      2            2   
1               Hugo Duro                28      9            0   
2       Roberto Férnandez                29      6            2   
3         Jorge de Frutos                27     10            1   
4          Gorka Guruzeta                26      6            1   
5          Borja Iglesias                26     11            2   
6            Mateo Joseph                28      2            2   
7           Ferran Jutglà                21      7            2   
8                    Kiké                28      6            1   
9             Adrián Liso                23      3            0   
10          Toni Martínez                28      7            3   
11               Rafa Mir                24      8            0   
12        Mikel Oyarzabal                26     12            3   
13  Isaac Palazón Camacho                28      3   

In [78]:
# Tenemos 18 delanteros con todas su métricas, ahora guardamos
df_delanteros.to_csv("datos/delanteros_final.csv", index=False)
print("Archivo guardado correctamente")

Archivo guardado correctamente


# 9. Preparación de Datos para llevarlos a MySQL Workbench

### 9.1 Preparación de la tabla de porteros.

In [79]:
# Ahora haremos una nueva limpieza de todas las tablas en cuanto a palabras con tildes, ñ y demas caracteres
# para que las tablas puedan ser manejadas correctamente por MySQLWORKBENCH
import pandas as pd

df_porteros = pd.read_csv("datos/porteros_final.csv")

df_porteros.to_csv("datos/porteros_mysql.csv", index=False, encoding='utf-8-sig')

print("Archivo guardado correctamente")

Archivo guardado correctamente


### 9.2 Limpieza y preparación de la tabla laterales derechos.

In [80]:
# Necesitamos hacer correciones en la tabla de laterales derechos antes de llevar a mysql
df_lat_der = pd.read_csv("datos/laterales_derechos_final.csv")

print(df_lat_der.shape)
print(df_lat_der['jugador'].tolist())

(17, 12)
['Jesús Areso', 'Héctor Bellerín', 'José Angel Carmona', 'Sergio Carreira', 'Kiko Femenía', 'Juan Antonio Iglesias', 'Álex Jiménez', 'Marcos Llorente', 'Pablo Maffeo', 'Arnau Martinez', 'Óscar Mingueza', 'Pau Navarro', 'Álvaro Núñez', 'Pedro Porro', 'Hugo Rincón', 'Juanlu Sánchez', 'Nacho Vidal']


In [81]:
# Remplazamos los nombres de jugadores con carácteres especiales
df_lat_der['jugador'] = df_lat_der['jugador'].replace({
    'Jesús Areso': 'Jesus Areso',
    'Héctor Bellerín': 'Hector Bellerin',
    'Kiko Femenía': 'Kiko Femenia',
    'Álex Jiménez': 'Alex Jimenez',
    'Óscar Mingueza': 'Oscar Mingueza',
    'Álvaro Núñez': 'Alvaro Nunez',
    'Hugo Rincón': 'Hugo Rincon',
    'Juanlu Sánchez': 'Juanlu Sanchez'
})

# Revisamos columnas
print(df_lat_der.columns.tolist())

# Rellenamos NaN
df_lat_der = df_lat_der.fillna(0)

# Guardamos
df_lat_der.to_csv("datos/laterales_derechos_mysql.csv", index=False, encoding='utf-8')
print("Archivo guardado correctamente")

['jugador', 'nacionalidad', 'posicion', 'sub_posición', 'partidos_jugados', 'partidos_titular', 'minutos_disputados', 'partidos_completos', 'goles', 'asistencias', 'duelos_ganados', 'intercepciones']
Archivo guardado correctamente


In [82]:
# Vemos que la columna de subposición tiene tilde, corregimos y guardamos
df_lat_der = df_lat_der.rename(columns={'sub_posición': 'sub_posicion'})

df_lat_der.to_csv("datos/laterales_derechos_mysql.csv", index=False, encoding='utf-8')
print("Archivo guardado correctamente")

Archivo guardado correctamente


### 9.3 Limpieza y preparación de la tabla laterales izquierdos.

In [83]:
# Hacemos el mismo proceso para los laterales izquierdos
df_lat_izq = pd.read_csv("datos/laterales_izquierdos_final.csv")

print(df_lat_izq.shape)
print(df_lat_izq['jugador'].tolist())

(20, 12)
['Alejandro Balde', 'Yuri Berchiche', 'Abel Bretones', 'Hugo Bueno', 'Sergi Cardona', 'Álvaro Carreras', 'Pep Chavarría', 'Marc Cucurella', 'Sergio Gómez', 'Álex Grimaldo', 'Javi López', 'José Luis Gayà', 'Aarón Martín', 'Juan Miranda', 'Álex Moreno', 'Victor Parada', 'Alfonso Pedraza', 'Carlos Romero', 'Manu Sánchez', 'Álex Valle']


In [84]:
# Remplazamos lo nombres con carácteres especiales 
df_lat_izq['jugador'] = df_lat_izq['jugador'].replace({
    'Álvaro Carreras': 'Alvaro Carreras',
    'Pep Chavarría': 'Pep Chavarria',
    'Sergio Gómez': 'Sergio Gomez',
    'Álex Grimaldo': 'Alex Grimaldo',
    'Javi López': 'Javi Lopez',
    'José Luis Gayà': 'Jose Luis Gaya',
    'Aarón Martín': 'Aaron Martin',
    'Álex Moreno': 'Alex Moreno',
    'Manu Sánchez': 'Manu Sanchez',
    'Álex Valle': 'Alex Valle'
})
# Eliminamos la columna con acento y los valores NaN que puedan existir
df_lat_izq = df_lat_izq.rename(columns={'sub_posición': 'sub_posicion'})
df_lat_izq = df_lat_izq.fillna(0)

df_lat_izq.to_csv("datos/laterales_izquierdos_mysql.csv", index=False, encoding='utf-8')
print("Archivo guardado correctamente")

Archivo guardado correctamente


### 9.4 Limpieza y preparación de la tabla defensas centrales.

In [85]:
# Hacemos el mismo proceso con los defensas centrales
df_def_cen = pd.read_csv("datos/defensas_centrales_final.csv")

print(df_def_cen.shape)
print(df_def_cen['jugador'].tolist())

(34, 10)
['Marcos Alonso', 'Raúl Asencio', 'Marc Bartra', 'Pedro Bigas', 'Fernando Calero', 'Jonny Castro', 'Catena', 'Víctor Chust', 'Copete', 'David Costas', 'Pau Cubarsí', 'Eric García', 'Mario Gila', 'Mario Hermoso', 'Jorge Herrando', 'Dean Huijsen', 'Adrián de la Fuente', 'Aymeric Laporte', 'Robin Le Normand', 'Rafa Marín', 'Gerard Martín', 'Jon Martin', 'Unai Núñez', 'Jon Pacheco', 'Aitor Paredes', 'Antonio Raillo', 'Jacobo Ramón', 'Diego Rico', 'Javi Rodríguez', 'Kike Salas', 'César Tárrega', 'Pau Torres', 'Daniel Vivian', 'Igor Zubeldia']


In [86]:
# Remplazamos los nombres con carácteres especiales
df_def_cen['jugador'] = df_def_cen['jugador'].replace({
    'Raúl Asencio': 'Raul Asencio',
    'Víctor Chust': 'Victor Chust',
    'Pau Cubarsí': 'Pau Cubarsi',
    'Eric García': 'Eric Garcia',
    'Adrián de la Fuente': 'Adrian de la Fuente',
    'Rafa Marín': 'Rafa Marin',
    'Gerard Martín': 'Gerard Martin',
    'Unai Núñez': 'Unai Nunez',
    'Jacobo Ramón': 'Jacobo Ramon',
    'Javi Rodríguez': 'Javi Rodriguez',
    'César Tárrega': 'Cesar Tarrega'
})
# remplazamos la columna con acento y los valores NaN que puedan existir
df_def_cen = df_def_cen.rename(columns={'sub_posición': 'sub_posicion'})
df_def_cen = df_def_cen.fillna(0)

df_def_cen.to_csv("datos/defensas_centrales_mysql.csv", index=False, encoding='utf-8')
print("Archivo guardado correctamente")

Archivo guardado correctamente


### 9.5 Limpieza y preparación de la tabla mediocampistas.

In [87]:
# Hacemos el mismo proceso con los mediocampistas
df_meds = pd.read_csv("datos/mediocampistas_final.csv")

print(df_meds.shape)
print(df_meds['jugador'].tolist())

(48, 12)
['Marc Aguado', 'Carles Aleñá', 'Carlos Álvarez', 'Pablo Barrios', 'Adrian Bernabe', 'Antonio Blanco', 'Santi Comesaña', 'Sergi Darder', 'Pedro Díaz', 'Edu Expósito', 'Aleix Febas', 'Pablo Fornals', 'Iñigo Ruiz de Galarreta', 'Aleix García', 'Nicolás González', 'Urko González', 'Jon Gorrotxategi', 'Javier Guerra', 'Pablo Ibáñez', 'Mikel Jauregizar', 'Koke', 'Fermin López', 'Unai López', 'Pol Lozano', 'Pablo Marín', 'Iván Martín', 'Mario Martín', 'Pablo Martínez', 'Brais Méndez', 'Luis Milla', 'Jon Moncayola', 'Manu Morlanes', 'Dani Olmo', 'Aimar Oroz', 'Daniel Parejo', 'Pedri', 'Alberto Reina', 'Oriol Rey', 'Marc Roca', 'Rodri', 'Miguel Román', 'Oihan Sancet', 'Carlos Soler', 'Hugo Sotelo', 'Lucas Torró', 'Óscar Valentín', 'José Luis García Vayá', 'Martín Zubimendi']


In [88]:
# Remplazamos los nombres con caracteres especiales
df_meds['jugador'] = df_meds['jugador'].replace({
    'Carles Aleñá': 'Carles Alena',
    'Carlos Álvarez': 'Carlos Alvarez',
    'Santi Comesaña': 'Santi Comesana',
    'Pedro Díaz': 'Pedro Diaz',
    'Edu Expósito': 'Edu Exposito',
    'Iñigo Ruiz de Galarreta': 'Inigo Ruiz de Galarreta',
    'Aleix García': 'Aleix Garcia',
    'Nicolás González': 'Nicolas Gonzalez',
    'Urko González': 'Urko Gonzalez',
    'Pablo Ibáñez': 'Pablo Ibanez',
    'Fermin López': 'Fermin Lopez',
    'Unai López': 'Unai Lopez',
    'Pablo Marín': 'Pablo Marin',
    'Iván Martín': 'Ivan Martin',
    'Mario Martín': 'Mario Martin',
    'Pablo Martínez': 'Pablo Martinez',
    'Brais Méndez': 'Brais Mendez',
    'Miguel Román': 'Miguel Roman',
    'Lucas Torró': 'Lucas Torro',
    'Óscar Valentín': 'Oscar Valentin',
    'José Luis García Vayá': 'Jose Luis Garcia Vaya',
    'Martín Zubimendi': 'Martin Zubimendi'
})
# Remplazamos el nombre de la columna subposicion con tilde
# Remplazamos cualquier valor NaN por 0
df_meds = df_meds.rename(columns={'sub_posición': 'sub_posicion'})
df_meds = df_meds.fillna(0)

df_meds.to_csv("datos/mediocampistas_mysql.csv", index=False, encoding='utf-8')
print("Archivo guardado correctamente")

Archivo guardado correctamente


### 9.6 Limpieza y preparación de la tabla extremos.

In [89]:
# Hacemos el mismo proceso con los extremos
df_extremos = pd.read_csv("datos/extremos_final.csv")

print(df_extremos.shape)
print(df_extremos['jugador'].tolist())

(18, 12)
['Alex Baena', 'Ander Barrenetxea', 'Álex Berenguer', 'Álvaro García', 'Rubén García', 'Bryan Gil', 'Diego López', 'Pere Milla', 'Alberto Moleiro', 'Víctor Muñoz', 'Yeremi Pino', 'Luis Rioja', 'Jesus Rodríguez', 'Aitor Ruibal', 'Germán Valera', 'Jan Virgili', 'Nico Williams', 'Lamine Yamal']


In [90]:
# Remplazamos los nombres de jugadores con caracteres especiales
df_extremos['jugador'] = df_extremos['jugador'].replace({
    'Álex Berenguer': 'Alex Berenguer',
    'Álvaro García': 'Alvaro Garcia',
    'Rubén García': 'Ruben Garcia',
    'Diego López': 'Diego Lopez',
    'Víctor Muñoz': 'Victor Munoz',
    'Jesus Rodríguez': 'Jesus Rodriguez',
    'Germán Valera': 'German Valera'
})
# Remplazamos el nombre de la columna subposicion sin tilde
# Remplzamos culaquier valor NaN por 0
df_extremos = df_extremos.rename(columns={'sub_posición': 'sub_posicion'})
df_extremos = df_extremos.fillna(0)

df_extremos.to_csv("datos/extremos_mysql.csv", index=False, encoding='utf-8')
print("Archivo guardado correctamente")

Archivo guardado correctamente


### 9.7 Limpieza y preparación de la tabla Delanteros.

In [91]:
# Hacemos el mismo proceso con los delanteros
df_delanteros = pd.read_csv("datos/delanteros_final.csv")

print(df_delanteros.shape)
print(df_delanteros['jugador'].tolist())

(18, 12)
['Pablo Durán', 'Hugo Duro', 'Roberto Férnandez', 'Jorge de Frutos', 'Gorka Guruzeta', 'Borja Iglesias', 'Mateo Joseph', 'Ferran Jutglà', 'Kiké', 'Adrián Liso', 'Toni Martínez', 'Rafa Mir', 'Mikel Oyarzabal', 'Isaac Palazón Camacho', 'Isaac Romero', 'Iván Romero', 'Ferrán Torres', 'Bryan Zaragoza']


In [92]:
# Remplzamos los nombres de jugadores con caracteres especiales
df_delanteros['jugador'] = df_delanteros['jugador'].replace({
    'Pablo Durán': 'Pablo Duran',
    'Roberto Férnandez': 'Roberto Fernandez',
    'Ferran Jutglà': 'Ferran Jutgla',
    'Kiké': 'Kike',
    'Adrián Liso': 'Adrian Liso',
    'Toni Martínez': 'Toni Martinez',
    'Mikel Oyarzabal': 'Mikel Oyarzabal',
    'Isaac Palazón Camacho': 'Isaac Palazon Camacho',
    'Iván Romero': 'Ivan Romero',
    'Ferrán Torres': 'Ferran Torres'})
# Eliminamos el nombre de la columna con tilde
# remplazamos cualquier valor NaN que pueda existir por cero
df_delanteros = df_delanteros.rename(columns={'sub_posición': 'sub_posicion'})
df_delanteros = df_delanteros.fillna(0)

df_delanteros.to_csv("datos/delanteros_mysql.csv", index=False, encoding='utf-8')
print("Archivo guardado correctamente")

Archivo guardado correctamente
